# Qwen3.5 ハンズオン: vLLM + GPU で画像 + テキスト推論

このNotebookでは、**`Qwen/Qwen3.5-2B` を vLLM 上で GPU 推論**します。

前回のNotebookは単に `.py` を叩いていたので、今回は **実際に動いた Python コードをそのままセルに分解** して、Notebookとして読める形に直しています。

## 0. まず確認

- モデル: `Qwen/Qwen3.5-2B`
- 方式: `vLLM` の offline inference
- 入力: **画像 + テキスト**
- 実行先: **GPU**

このNotebookは **`Python (.venv-vllm qwen35)` kernel** で開く前提です。

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import requests

MODEL_ID = 'Qwen/Qwen3.5-2B'
IMAGE_URL = 'https://qianwen-res.oss-accelerate.aliyuncs.com/Qwen3.5/demo/RealWorld/RealWorld-04.png'
IMAGE_PATH = Path('../data/qwen35_demo.png')
IMAGE_PATH.parent.mkdir(parents=True, exist_ok=True)

if not IMAGE_PATH.exists():
    response = requests.get(IMAGE_URL, timeout=60)
    response.raise_for_status()
    IMAGE_PATH.write_bytes(response.content)

image = Image.open(IMAGE_PATH).convert('RGB')
plt.figure(figsize=(8, 5))
plt.imshow(image)
plt.axis('off')
plt.show()
print(IMAGE_PATH)

## 1. 画像 + テキストのメッセージを作る

Qwen3.5 は `-VL` なしでもそのままマルチモーダルなので、`AutoProcessor` で chat template を作り、画像とテキストを一緒に渡します。

In [ ]:
from transformers import AutoProcessor

messages = [
    {
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text', 'text': 'Where is this? Describe the scene briefly.'},
        ],
    }
]

processor = AutoProcessor.from_pretrained(MODEL_ID)
prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt[:500])

## 2. vLLM エンジンを立ち上げる

ここが一番大事なセルです。
`LLM(...)` を作って、`multi_modal_data={'image': image}` を使う準備をします。

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(
    model=MODEL_ID,
    gpu_memory_utilization=0.85,
    max_model_len=8192,
    limit_mm_per_prompt={'image': 1},
)

sampling_params = SamplingParams(max_tokens=128, temperature=0.0)

## 3. 実際に生成する

In [ ]:
outputs = llm.generate(
    {
        'prompt': prompt,
        'multi_modal_data': {'image': image},
    },
    sampling_params=sampling_params,
)

answer = outputs[0].outputs[0].text
print(answer)

## 4. まとめ

このNotebookでやったこと:
- `AutoProcessor` で Qwen3.5 用の multimodal prompt を作る
- `vLLM` の `LLM(...)` を GPU 上で起動する
- `multi_modal_data` に画像を入れて、**画像 + テキスト推論**を行う

つまり、**Qwen3.5 は `-VL` なしでそのまま VLM として使える**、というのがここでのポイントです。